# Comprehensive MLPipelineHolder Example

This notebook demonstrates a broad set of `MLPipelineHolder` features in one place.

It shows how to:
- create a root pipeline and nested child pipelines
- use config-based gates with expected values
- use float priority branch groups
- use block-scoped `register_args(...)` and `register_kwargs(...)` helpers
- inspect configs, blocks, child pipelines, and priority groups
- run partial paths before a full run
- save and load the pipeline


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import shutil
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.mlpipelineholder import PipelineHandler


## Example functions and configs

The functions below are intentionally small so the pipeline structure is easy to follow.

In [ ]:
@dataclass
class ParentConfig:
    raw_value: int = 5
    multiplier: int = 3
    model_cls: str = "cls_a"
    verbose: bool = False


@dataclass
class ChildConfig:
    branch_label: str
    branch_lr: float


def create_seed(raw_value: int) -> int:
    return raw_value + 1


def create_training_inputs(seed: int) -> tuple[int, float, float, float]:
    return seed * 2, 0.1, 0.2, 0.3


def announce_seed(seed: int, logger, verbose: bool = False) -> None:
    logger.info(f"seed={seed}")
    if verbose:
        print(f"verbose-seed={seed}")


def training_loop(model_name: str, *score_parts: int, lr: float = 1.0, **weights: float) -> tuple[dict[str, float | str], str]:
    total = round(sum(score_parts) + sum(weights.values()) + lr, 3)
    metrics = {"model_name": model_name, "score": total}
    return metrics, f"{model_name}:{total}"


def publish_branch_result(branch_summary: str, logger) -> str:
    logger.result(branch_summary)
    return branch_summary


def finalize(branch_metrics: dict[str, float | str], multiplier: int) -> float:
    return float(branch_metrics["score"]) * multiplier


def summarize_branch(branch_report: str, logger) -> None:
    logger.print(f"branch-report={branch_report}")


## Build a reusable child pipeline

Each branch has its own config and gate, but it can still read parent outputs and config values.

In [ ]:
def build_training_branch(
    registration_name: str,
    expected_model_cls: str,
    branch_label: str,
    branch_lr: float,
    project_root: Path,
) -> PipelineHandler:
    child = PipelineHandler(
        registration_name=registration_name,
        configuration=ChildConfig(branch_label=branch_label, branch_lr=branch_lr),
        local_folder_path=project_root / registration_name,
    )
    child.add_gate_block("model_cls", expected_model_cls)

    training = child.add_block("model_training", 30.0, forced=True)
    training.register_args("training_args", ("seed", "feature_size"), forced=True)
    training.register_kwargs(
        "training_kwargs",
        {
            "beta": "beta",
            "max_error_weight": "max_error_weight",
            "neighbour_similarity_weight": "neighbour_similarity_weight",
        },
        forced=True,
    )
    training.register_function(
        training_loop,
        ["branch_metrics", "branch_summary"],
        save_to_disk=["branch_metrics"],
        param_mapping={"model_name": "branch_label", "lr": "branch_lr"},
        var_pos_name="training_args",
        var_kw_name="training_kwargs",
        forced=True,
    )

    reporting = child.add_block("branch_reporting", 31.0, forced=True)
    reporting.register_function(publish_branch_result, ["branch_report"], forced=True)
    summary = child.add_block("branch_summary", 32.0, forced=True)
    summary.register_function(summarize_branch, None, forced=True)
    return child


## Build the root pipeline

The root pipeline prepares shared inputs, logs progress, branches into model-specific children, and then finalizes the result.

In [ ]:
def build_pipeline(project_root: Path) -> PipelineHandler:
    pipeline = PipelineHandler(
        registration_name="comprehensive-pipeline",
        configuration=ParentConfig(),
        local_folder_path=project_root,
    )

    setup = pipeline.add_block("setup", 10.0, forced=True)
    setup.register_function(create_seed, ["seed"], forced=True)

    prep = pipeline.add_block("prepare_training_inputs", 20.0, forced=True)
    prep.register_function(
        create_training_inputs,
        ["feature_size", "beta", "max_error_weight", "neighbour_similarity_weight"],
        forced=True,
    )

    logging_block = pipeline.add_block("logging", 25.0, forced=True)
    logging_block.register_function(announce_seed, None, forced=True)

    model_a = build_training_branch("model_a_training_pipeline", "cls_a", "model_a", 0.75, project_root)
    model_b = build_training_branch("model_b_training_pipeline", "cls_b", "model_b", 1.25, project_root)
    pipeline.add_child_pipeline(model_a, 40.3, forced=True)
    pipeline.add_child_pipeline(model_b, 40.4, forced=True)

    final = pipeline.add_block("finalize", 50.0, forced=True)
    final.register_function(finalize, ["final_score"], forced=True)
    return pipeline


## Create a clean example run folder

The notebook uses `examples/example_run/` as the pipeline root so it keeps all generated files inside the examples area.

In [ ]:
project_root = Path.cwd() / "example_run"
if project_root.exists():
    shutil.rmtree(project_root)

pipeline = build_pipeline(project_root)
pipeline.set_log_level("info")
pipeline.set_print_capture_mode("tee")
pipeline


## Inspect helper APIs before running

These helpers are useful when interactively exploring or debugging a pipeline.

In [ ]:
names, active = pipeline.get_priority_group(40)
model_a = pipeline.get_child_pipeline("model_a_training_pipeline")
model_training_block = model_a.get_block("model_training")

print(f"priority_group_40={names}, active={active}")
print(f"root_config={pipeline.get_full_config()}")
print(f"child_visible_config={model_a.get_full_config()}")
print(f"model_training_block={model_training_block.registration_name}")


## Warm up only part of the pipeline

This demonstrates nested path targeting with `run_until(...)`.

In [ ]:
warmup = pipeline.run_until("model_a_training_pipeline", "model_training")
print(f"warmup_run_status={warmup.status}")
print(f"warmup_nodes={warmup.executed_blocks}")


## Run the full pipeline

Because `model_cls='cls_a'`, the `model_a_training_pipeline` branch runs and the `model_b_training_pipeline` branch is shown as muted in the chart.

In [ ]:
run = pipeline.run_all()
print(f"run_status={run.status}")
print(f"executed_nodes={run.executed_blocks}")
print(f"branch_summary={pipeline.get_value('branch_summary')}")
print(f"branch_metrics={pipeline.get_value('branch_metrics')}")
print(f"final_score={pipeline.get_value('final_score')}")
print(f"result_history={pipeline.get_result_history()}")


## Visualize the pipeline

The chart includes nested children, float-priority groups, helper-based variadics, and gate state.

In [ ]:
print(pipeline.describe_pipeline())


## Save and load the pipeline

This demonstrates the current save/load path along with optional log export.

In [ ]:
pipeline.save_pipeline(save_log_to_file=project_root / "saved_example.log")
loaded = PipelineHandler.load_pipeline(project_root)
loaded_run = loaded.run_all()
print(f"loaded_run_status={loaded_run.status}")
print(f"loaded_final_score={loaded.get_value('final_score')}")


## Reset the root gate and inspect again

The helper APIs also make it easy to modify and inspect pipelines interactively.

In [ ]:
loaded.reset_gate_block()
print(loaded.describe_pipeline())
